# 🏗️ Notebook 06: Transformer Architecture

This notebook assembles a complete transformer block from the components we built in previous notebooks: multi-head attention, feed-forward networks, layer normalization, and residual connections.

**Prerequisites:** Notebooks 01-05

## ✅ Environment Validation

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Transformer Block Diagram

```
Input (batch, seq_len, d_model)
  │
  ├──────────────────────┐
  │                      │ (residual)
  ▼                      │
RMSNorm                  │
  │                      │
  ▼                      │
Multi-Head Attention      │
  │                      │
  ▼                      │
  + ◄────────────────────┘
  │
  ├──────────────────────┐
  │                      │ (residual)
  ▼                      │
RMSNorm                  │
  │                      │
  ▼                      │
Feed-Forward (SwiGLU)    │
  │                      │
  ▼                      │
  + ◄────────────────────┘
  │
  ▼
Output (batch, seq_len, d_model)
```

💡 Modern LLMs (LLaMA, Gemma) use **pre-norm** (normalize before sublayer) and **RMSNorm** (simpler than LayerNorm).

## Feed-Forward Network (SwiGLU)

The FFN in modern LLMs uses the SwiGLU activation: `FFN(x) = (xW₁ ⊙ SiLU(xW_gate)) W₂`

This is a gated linear unit where SiLU acts as the gate. It's used in LLaMA, Gemma, and Mistral.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import math
import numpy as np
import matplotlib.pyplot as plt

class SwiGLUFFN(nn.Module):
    """Feed-forward network with SwiGLU activation (used in LLaMA, Gemma)."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)     # up projection
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)  # gate projection
        self.w2 = nn.Linear(d_ff, d_model, bias=False)      # down projection
    
    def __call__(self, x: mx.array) -> mx.array:
        # x: (batch, seq_len, d_model)
        return self.w2(nn.silu(self.w_gate(x)) * self.w1(x))  # (batch, seq_len, d_model)

# Demo
d_model, d_ff = 32, 128  # d_ff is typically 4x d_model
ffn = SwiGLUFFN(d_model, d_ff)
x = mx.random.normal((1, 4, d_model))  # (batch=1, seq=4, d=32)
out = ffn(x)
mx.eval(out)
print(f'Input:  {x.shape}')   # (1, 4, 32)
print(f'Output: {out.shape}')  # (1, 4, 32)
print(f'FFN shape: d_model={d_model} → d_ff={d_ff} → d_model={d_model}')

## Layer Normalization: LayerNorm vs RMSNorm

**LayerNorm**: Normalizes to zero mean and unit variance, then applies scale (γ) and shift (β).

**RMSNorm**: Only normalizes by root-mean-square — no mean subtraction, no shift. Simpler and ~10-15% faster.

💡 Modern LLMs use RMSNorm with **pre-norm** placement (normalize before the sublayer, not after).

In [ ]:
# RMSNorm from scratch
class RMSNormManual(nn.Module):
    def __init__(self, dims: int, eps: float = 1e-6):
        super().__init__()
        self.weight = mx.ones((dims,))
        self.eps = eps
    
    def __call__(self, x: mx.array) -> mx.array:
        rms = mx.sqrt(mx.mean(x * x, axis=-1, keepdims=True) + self.eps)
        return (x / rms) * self.weight

# Compare with MLX built-in
x = mx.random.normal((1, 4, 32))
manual_norm = RMSNormManual(32)
builtin_norm = nn.RMSNorm(32)

out_manual = manual_norm(x)
out_builtin = builtin_norm(x)
mx.eval(out_manual, out_builtin)

diff = mx.max(mx.abs(out_manual - out_builtin)).item()
print(f'Max diff between manual and built-in RMSNorm: {diff:.2e}')
print(f'Match: {diff < 1e-5} ✅')

## Residual Connections

Residual connections (`x + sublayer(norm(x))`) solve the vanishing gradient problem in deep networks. The gradient flows directly through the `+` operation, bypassing the sublayer.

💡 **Intuition**: Think of residuals as a highway bypass — information can skip layers that aren't helpful. Each sublayer only needs to learn the *difference* between its input and the ideal output, not the full transformation from scratch. If a layer has nothing useful to add, its output can be near-zero and the input passes through unchanged. This means deeper networks can never be *worse* than shallower ones, because extra layers can simply learn to do nothing.

🎯 **Interview tip**: Without residuals, a 32-layer transformer would be nearly impossible to train. The gradient signal would vanish exponentially with depth.

In [ ]:
# Demonstrate residual connection pattern
x = mx.random.normal((1, 4, 32))
norm = nn.RMSNorm(32)
ffn = SwiGLUFFN(32, 128)

# Pre-norm residual: x + sublayer(norm(x))
residual_out = x + ffn(norm(x))
mx.eval(residual_out)

print(f'Input shape:    {x.shape}')
print(f'Residual output: {residual_out.shape}')
print(f'Shape preserved: {x.shape == residual_out.shape} ✅')
print(f'\n💡 The residual connection ensures the output is always at least as good as the input.')

## Complete TransformerBlock

Now we assemble everything: attention → add & norm → FFN → add & norm.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = math.sqrt(self.d_head)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def __call__(self, x, mask=None):
        B, T, D = x.shape
        Q = self.W_q(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        K = self.W_k(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        V = self.W_v(x).reshape(B, T, self.n_heads, self.d_head).transpose(0, 2, 1, 3)
        scores = (Q @ K.transpose(0, 1, 3, 2)) / self.scale
        if mask is not None:
            scores = mx.where(mask == 0, mx.array(float('-inf')), scores)
        weights = mx.softmax(scores, axis=-1)
        out = (weights @ V).transpose(0, 2, 1, 3).reshape(B, T, D)
        return self.W_o(out)

class TransformerBlock(nn.Module):
    """Single transformer block: attention + FFN with pre-norm residuals."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.attn_norm = nn.RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ffn_norm = nn.RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model, d_ff)
    
    def __call__(self, x: mx.array, mask: mx.array = None) -> mx.array:
        # x: (batch, seq_len, d_model)
        x = x + self.attn(self.attn_norm(x), mask)  # attention + residual
        x = x + self.ffn(self.ffn_norm(x))           # FFN + residual
        return x  # (batch, seq_len, d_model)

# Test
block = TransformerBlock(d_model=64, n_heads=4, d_ff=256)
x = mx.random.normal((2, 8, 64))  # (batch=2, seq=8, d=64)
mask = mx.tril(mx.ones((8, 8)))
out = block(x, mask)
mx.eval(out)
print(f'Input:  {x.shape}')   # (2, 8, 64)
print(f'Output: {out.shape}')  # (2, 8, 64)
print(f'Shape preserved through block: ✅')

## Stacking N Blocks: TransformerStack

A full transformer is just N identical blocks stacked. The output shape is preserved through all layers.

In [ ]:
class TransformerStack(nn.Module):
    def __init__(self, n_layers: int, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.layers = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.final_norm = nn.RMSNorm(d_model)
    
    def __call__(self, x: mx.array, mask: mx.array = None) -> mx.array:
        for layer in self.layers:
            x = layer(x, mask)
        return self.final_norm(x)

# Count parameters
def count_params(model):
    total = 0
    for k, v in model.parameters().items():
        if isinstance(v, dict):
            for kk, vv in v.items():
                if hasattr(vv, 'size'): total += vv.size
                elif isinstance(vv, dict):
                    for kkk, vvv in vv.items():
                        if hasattr(vvv, 'size'): total += vvv.size
                        elif isinstance(vvv, dict):
                            for kkkk, vvvv in vvv.items():
                                if hasattr(vvvv, 'size'): total += vvvv.size
        elif isinstance(v, list):
            for item in v:
                if isinstance(item, dict):
                    for kk, vv in item.items():
                        if hasattr(vv, 'size'): total += vv.size
                        elif isinstance(vv, dict):
                            for kkk, vvv in vv.items():
                                if hasattr(vvv, 'size'): total += vvv.size
                                elif isinstance(vvv, dict):
                                    for kkkk, vvvv in vvv.items():
                                        if hasattr(vvvv, 'size'): total += vvvv.size
        elif hasattr(v, 'size'): total += v.size
    return total

# Test with 4 layers
stack = TransformerStack(n_layers=4, d_model=64, n_heads=4, d_ff=256)
x = mx.random.normal((2, 8, 64))
mask = mx.tril(mx.ones((8, 8)))
out = stack(x, mask)
mx.eval(out)

n_params = count_params(stack)
print(f'Stack: 4 layers, d_model=64, n_heads=4, d_ff=256')
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')
print(f'Total parameters: {n_params:,}')
print(f'\n🎯 Output shape is preserved through all {4} layers ✅')

## 🔬 Backpropagation Walkthrough

Let's trace gradient flow through a single transformer block step by step. Understanding where gradients flow (and where they can vanish) is essential for interview discussions.

💡 In a pre-norm block `x + sublayer(norm(x))`, the gradient always has an **identity path** through the residual connection. This prevents vanishing gradients even in very deep stacks.

```
∂L/∂x_l = ∂L/∂x_{l+1} × (I + ∂sublayer(norm(x_l))/∂x_l)
                              ↑ identity path (never vanishes)
```

🎯 **Interview tip**: "Pre-norm ensures the gradient at any layer is at least as large as the gradient at the next layer, because the residual connection provides a direct gradient highway."

In [ ]:
from utils.transformer_analysis import backprop_walkthrough

# Step-by-step gradient norms through a single transformer block
result = backprop_walkthrough(d_model=64, n_heads=4)

print(f'Loss: {result["loss"]:.6f}')
print(f'\n🔬 Gradient norms per component:')
for component, norm in result['component_grad_norms'].items():
    bar = '█' * int(norm * 20)
    print(f'  {component:25s}: {norm:.6f}  {bar}')

print(f'\n💡 The attention and FFN components receive comparable gradient signal')
print(f'   thanks to the residual connections preserving gradient flow.')

## ⚡ Activation Function Comparison

Modern LLMs have evolved through several activation functions. Let's compare them side by side:

| Activation | Formula | Used In |
|---|---|---|
| ReLU | max(0, x) | Early transformers |
| GELU | x × Φ(x) | BERT, GPT-2 |
| SiLU (Swish) | x × σ(x) | LLaMA, Mistral |
| SwiGLU | (xW₁) ⊙ SiLU(xW_gate) | LLaMA, Gemma |
| GeGLU | (xW₁) ⊙ GELU(xW_gate) | Some T5 variants |

💡 Gated variants (SwiGLU, GeGLU) use **two** projections — one for the value and one for the gate. This adds parameters but improves expressiveness.

⚠️ SwiGLU has 3 weight matrices (W₁, W_gate, W₂) vs 2 for standard FFN (W₁, W₂), so the d_ff is typically reduced by 2/3 to keep parameter count similar.

In [ ]:
from utils.transformer_analysis import ActivationComparison
import numpy as np
import mlx.core as mx

# Compare activations on the same input range
x_np = np.linspace(-4, 4, 200).astype(np.float32)
x_mx = mx.array(x_np)

activations = ActivationComparison.compute(x_mx)
derivatives = ActivationComparison.compute_derivatives_np(x_np)
fig = ActivationComparison.plot(x_np, activations, derivatives)
fig.savefig('data/activation_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print('🎯 Key observations:')
print('  • ReLU has a dead zone (derivative = 0 for x < 0)')
print('  • GELU/SiLU are smooth approximations of ReLU — no dead neurons')
print('  • SwiGLU/GeGLU are gated: output = linear × gate(linear)')

## 🔄 Normalization Comparison & Gradient Flow

Three normalization strategies used in transformers:

| Method | Formula | Advantage |
|---|---|---|
| LayerNorm | (x - μ) / √(σ² + ε) × γ + β | Original transformer |
| RMSNorm | x / √(mean(x²) + ε) × γ | ~15% faster, no mean subtraction |
| DeepNorm | LayerNorm(α·x + sublayer(x)) | Enables training 1000+ layer transformers |

💡 **Pre-norm** (normalize before sublayer) is now standard in all modern LLMs. It provides more stable gradient flow than **post-norm** (normalize after sublayer).

🎯 **Interview tip**: "Pre-norm maintains higher mean activation gradient norms across layers because the residual path provides a direct gradient highway that bypasses the normalization."

In [ ]:
from utils.transformer_analysis import NormalizationComparison, gradient_flow_analysis
import numpy as np
import mlx.core as mx

# Compare normalizations
x_np = np.sort(np.random.randn(64).astype(np.float32))
x_mx = mx.array(x_np)
norms = NormalizationComparison.compute(x_mx)
fig_norms = NormalizationComparison.plot(x_np, norms)
plt.show()

# Gradient flow: pre-norm vs post-norm
result = gradient_flow_analysis(depth=8, d_model=64, seq_len=4, batch=2, n_trials=10)
result['figure'].savefig('data/gradient_flow.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Pre-norm  mean gradient: {result["pre_mean_grad"]:.4f}  (CV={result["pre_norm_cv"]:.4f})')
print(f'Post-norm mean gradient: {result["post_mean_grad"]:.4f}  (CV={result["post_norm_cv"]:.4f})')
print(f'\n⚡ Pre-norm maintains {result["pre_mean_grad"]/result["post_mean_grad"]:.1%} of post-norm gradient magnitude')
print(f'   → Stronger learning signal at every layer')

## 📊 Parameter Counter & Memory Estimation

Understanding where parameters live in a transformer is crucial for model design and optimization.

🎯 **Interview tip**: For a standard transformer, the parameter breakdown is roughly:
- **Embedding**: vocab_size × d_model (often the largest single component)
- **Attention**: 4 × d_model² per layer (Q, K, V, O projections)
- **FFN**: 3 × d_model × d_ff per layer for gated variants (SwiGLU)
- **Normalization**: negligible (just d_model per norm layer)

In [ ]:
from utils.transformer_analysis import ParameterCounter, TransformerConfig

# Count parameters for a LLaMA-like config
config = TransformerConfig(
    d_model=768, n_heads=12, n_kv_heads=4,
    n_layers=12, d_ff=3072, vocab_size=32000,
    activation='swiglu', norm_type='rmsnorm'
)

counts = ParameterCounter.count(config)
memory_f32 = ParameterCounter.estimate_memory(config, 'float32')
memory_f16 = ParameterCounter.estimate_memory(config, 'float16')
memory_int4 = ParameterCounter.estimate_memory(config, 'int4')

print(f'TransformerConfig: d={config.d_model}, layers={config.n_layers}, heads={config.n_heads}')
print(f'\n📊 Parameter breakdown:')
for comp in ['embedding', 'attention', 'ffn', 'normalization']:
    pct = counts[comp] / counts['total'] * 100
    print(f'  {comp:15s}: {counts[comp]:>12,}  ({pct:5.1f}%)')
print(f'  {"total":15s}: {counts["total"]:>12,}')

print(f'\n💾 Memory estimates:')
print(f'  float32: {memory_f32["total"] / 1e9:.2f} GB')
print(f'  float16: {memory_f16["total"] / 1e9:.2f} GB')
print(f'  int4:    {memory_int4["total"] / 1e9:.2f} GB')

# Visualization
fig = ParameterCounter.plot(config, 'float16')
plt.show()

## 🎲 Weight Initialization: Xavier vs Kaiming

Proper weight initialization is critical for training stability. The wrong initialization can cause gradients to explode or vanish from the very first step.

| Strategy | Formula | Best For |
|---|---|---|
| Xavier (Glorot) | W ~ U(-√(6/(fan_in+fan_out)), √(6/(fan_in+fan_out))) | Sigmoid, Tanh, linear layers |
| Kaiming (He) | W ~ N(0, √(2/fan_in)) | ReLU-family activations |

💡 **Why it matters**: Xavier assumes the activation is linear (preserves variance). Kaiming accounts for the fact that ReLU zeros out half the activations, so it doubles the variance to compensate.

⚠️ Most modern LLMs use a scaled version of Xavier or Kaiming initialization, often with an additional 1/√(2·n_layers) factor to account for the residual stream accumulation across layers.

🎯 **Interview tip**: "GPT-style models typically scale the output projection of each residual sublayer by 1/√(n_layers) to prevent the residual stream variance from growing linearly with depth."

In [ ]:
from utils.transformer_analysis import plot_init_distributions

fig = plot_init_distributions(d=256)
plt.show()

print('💡 Notice how Kaiming has wider spread than Xavier — this compensates')
print('   for the variance reduction caused by ReLU zeroing negative values.')

## 🧪 Try It Yourself

Test your understanding of the transformer block:

1. **Residual connections**: Comment out the residual connection in the transformer block (the `+ x` part). Run a forward pass. What happens to the output? Why?

2. **Pre-norm vs post-norm**: Move the RMSNorm to AFTER the attention/FFN instead of before. Does the output change? (Hint: yes, and it matters for training stability.)

3. **Parameter counting**: How many parameters does a single transformer block have with d_model=4096, n_heads=32, d_ff=11008? Calculate by hand, then verify with code.

## 📜 History & Alternatives

### Activation Functions Evolution

| Year | Activation | Key Insight | Used In |
|---|---|---|---|
| 2010 | **ReLU** | Simple, fast, avoids vanishing gradient | AlexNet, early transformers |
| 2016 | **GELU** (Hendrycks & Gimpel) | Smooth approximation of ReLU, stochastic regularization | BERT, GPT-2 |
| 2017 | **SiLU/Swish** (Ramachandran et al.) | Self-gated: x × σ(x), found via NAS | EfficientNet |
| 2020 | **SwiGLU** (Shazeer) | Gated linear unit with SiLU gate, +1% quality | PaLM, LLaMA, Gemma |
| 2020 | **GeGLU** (Shazeer) | Gated linear unit with GELU gate | Some T5 variants |

### Normalization Evolution

| Year | Method | Key Insight | Used In |
|---|---|---|---|
| 2015 | **BatchNorm** (Ioffe & Szegedy) | Normalize across batch dimension | CNNs (not used in LLMs) |
| 2016 | **LayerNorm** (Ba et al.) | Normalize across feature dimension, batch-independent | Original Transformer, BERT |
| 2019 | **RMSNorm** (Zhang & Sennrich) | Drop mean subtraction, ~15% faster | LLaMA, Gemma, Mistral |
| 2022 | **DeepNorm** (Wang et al.) | Scale residual by α, enables 1000+ layers | DeepNet |

### Pre-Norm vs Post-Norm

The original Transformer (2017) used **post-norm**: `norm(x + sublayer(x))`. This requires careful learning rate warmup and can be unstable.

**Pre-norm** (`x + sublayer(norm(x))`) was shown to be more stable by Xiong et al. (2020) in "On Layer Normalization in the Transformer Architecture". All modern LLMs (GPT-3+, LLaMA, Gemma, Mistral) use pre-norm.

⚡ The tradeoff: post-norm can achieve slightly better final performance with careful tuning, but pre-norm is much easier to train and scales better to deep networks.

## Summary

We assembled and deeply analyzed a complete transformer block:
- **Multi-head attention** (from Notebook 05)
- **SwiGLU FFN** (gated feed-forward, used in LLaMA/Gemma)
- **RMSNorm** (simpler, faster than LayerNorm)
- **Residual connections** (enable deep stacking)
- 🔬 **Backpropagation walkthrough** — gradient flow through each component
- ⚡ **Activation comparison** — ReLU → GELU → SiLU → SwiGLU → GeGLU
- 🔄 **Normalization comparison** — LayerNorm vs RMSNorm vs DeepNorm
- 📊 **Parameter counting** — per-component breakdown and memory estimation
- 🎲 **Weight initialization** — Xavier vs Kaiming strategies
- 📜 **History** — evolution of activations and normalizations

**Next:** Notebook 07 — Building a GPT from Scratch